## Instalación de librerias

In [3]:
import pandas as pd
import sqlite3

## Cargar dataset final

In [4]:
# Cargar dataset transformado
df = pd.read_csv('../results/dataset_final.csv')

print(df.shape)
df.head()

(116320, 11)


,cve_id,descripcion,fecha,cvss_score,severidad,tiene_exploit,criticidad,anio_actual,antiguedad,categoria,score_prioridad
0,CVE-2023-3710,Improper Input Validation vulnerability in Hon...,2023-09-12 20:15:09.387,9.9,CRITICAL,1,critica,2026,3,Other,10.83
1,CVE-2023-36355,TP-Link TL-WR940N V4 was discovered to contain...,2023-06-22 20:15:09.733,9.9,CRITICAL,1,critica,2026,3,Buffer Overflow,10.83
2,CVE-2023-27823,An authentication bypass in Optoma 1080PSTX C0...,2023-05-12 14:15:09.727,9.8,CRITICAL,1,critica,2026,3,Other,10.76
3,CVE-2023-1934,"The PnPSCADA system, a product of SDG Technolo...",2023-05-12 14:15:09.653,9.8,CRITICAL,1,critica,2026,3,SQL Injection,10.76
4,CVE-2023-23163,Art Gallery Management System Project v1.0 was...,2023-02-10 20:15:53.760,9.8,CRITICAL,1,critica,2026,3,SQL Injection,10.76


## Base de datos SQLite

In [5]:
# Crear conexión a base de datos (se crea automáticamente)
conn = sqlite3.connect('../results/vulnerabilidades.db')

print("Creada correctamente")

Creada correctamente


## Cargar datos a la base de datos

In [6]:
# Cargar DataFrame a tabla SQL
df.to_sql('vulnerabilidades', conn, if_exists='replace', index=False)

print("Cargado correctamente")

Cargado correctamente


## Validar carga

In [7]:
# Ejecutar consulta simple
query = "SELECT * FROM vulnerabilidades LIMIT 5"

df_sql = pd.read_sql(query, conn)

df_sql

,cve_id,descripcion,fecha,cvss_score,severidad,tiene_exploit,criticidad,anio_actual,antiguedad,categoria,score_prioridad
0,CVE-2023-3710,Improper Input Validation vulnerability in Hon...,2023-09-12 20:15:09.387,9.9,CRITICAL,1,critica,2026,3,Other,10.83
1,CVE-2023-36355,TP-Link TL-WR940N V4 was discovered to contain...,2023-06-22 20:15:09.733,9.9,CRITICAL,1,critica,2026,3,Buffer Overflow,10.83
2,CVE-2023-27823,An authentication bypass in Optoma 1080PSTX C0...,2023-05-12 14:15:09.727,9.8,CRITICAL,1,critica,2026,3,Other,10.76
3,CVE-2023-1934,"The PnPSCADA system, a product of SDG Technolo...",2023-05-12 14:15:09.653,9.8,CRITICAL,1,critica,2026,3,SQL Injection,10.76
4,CVE-2023-23163,Art Gallery Management System Project v1.0 was...,2023-02-10 20:15:53.760,9.8,CRITICAL,1,critica,2026,3,SQL Injection,10.76


## Consultas preguntas estratégicas/negocio

### 1. ¿Qué porcentaje de vulnerabilidades catalogadas como críticas cuentan realmente con exploit público disponible?

In [8]:
query = """
SELECT 
    (SUM(CASE WHEN tiene_exploit = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) 
    AS porcentaje_criticas_con_exploit
FROM vulnerabilidades
WHERE criticidad = 'critica'
"""

pd.read_sql(query, conn)

,porcentaje_criticas_con_exploit
0,1.212711


### 2. ¿Existen vulnerabilidades con puntaje medio o bajo que presenten evidencia de explotación activa según el registro Exploit Database?

In [9]:
query = """
SELECT cve_id, criticidad, cvss_score
FROM vulnerabilidades
WHERE criticidad IN ('baja', 'media')
AND tiene_exploit = 1
LIMIT 10
"""

pd.read_sql(query, conn)

,cve_id,criticidad,cvss_score
0,CVE-2023-34927,media,6.5
1,CVE-2023-34927,media,6.5
2,CVE-2023-32750,media,6.5
3,CVE-2023-34927,media,6.5
4,CVE-2023-34096,media,6.5
5,CVE-2023-33145,media,6.5
6,CVE-2023-24626,media,6.5
7,CVE-2023-27167,media,6.5
8,CVE-2024-35133,media,6.8
9,CVE-2024-23922,media,6.8


### 3. ¿Qué categorías de vulnerabilidad concentran mayor frecuencia o mayor disponibilidad de exploits?

In [10]:
query = """
SELECT 
    categoria,
    COUNT(*) as total_vulnerabilidades,
    SUM(tiene_exploit) as total_exploits
FROM vulnerabilidades
GROUP BY categoria
ORDER BY total_vulnerabilidades DESC
"""

pd.read_sql(query, conn)

,categoria,total_vulnerabilidades,total_exploits
0,Other,64212,259
1,RCE,22462,125
2,XSS,13051,47
3,SQL Injection,10030,66
4,Buffer Overflow,6565,13


### 4. ¿Qué criterios permiten construir un ranking alternativo que optimice la asignación de recursos limitados?

In [11]:
query = """
SELECT 
    cve_id,
    cvss_score,
    tiene_exploit,
    antiguedad,
    score_prioridad
FROM vulnerabilidades
ORDER BY score_prioridad DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,cve_id,cvss_score,tiene_exploit,antiguedad,score_prioridad
0,CVE-2023-3710,9.9,1,3,10.83
1,CVE-2023-36355,9.9,1,3,10.83
2,CVE-2023-27823,9.8,1,3,10.76
3,CVE-2023-1934,9.8,1,3,10.76
4,CVE-2023-23163,9.8,1,3,10.76
5,CVE-2023-39115,9.8,1,3,10.76
6,CVE-2023-6019,9.8,1,3,10.76
7,CVE-2023-27100,9.8,1,3,10.76
8,CVE-2023-28343,9.8,1,3,10.76
9,CVE-2023-23162,9.8,1,3,10.76


### 5. ¿Qué brechas existen entre la priorización basada únicamente en CVSS y un enfoque basado en integración de datos múltiples?

In [12]:
query = """
SELECT 
    AVG(cvss_score) as promedio_cvss,
    AVG(score_prioridad) as promedio_modelo
FROM vulnerabilidades
"""

pd.read_sql(query, conn)

,promedio_cvss,promedio_modelo
0,6.695938,5.18579


In [13]:
query = """
SELECT 
    'CVSS' as metodo,
    AVG(tiene_exploit)*100 as porcentaje_exploit
FROM (
    SELECT * FROM vulnerabilidades
    ORDER BY cvss_score DESC
    LIMIT 10
)

UNION

SELECT 
    'MODELO' as metodo,
    AVG(tiene_exploit)*100 as porcentaje_exploit
FROM (
    SELECT * FROM vulnerabilidades
    ORDER BY score_prioridad DESC
    LIMIT 10
)
"""

pd.read_sql(query, conn)

,metodo,porcentaje_exploit
0,CVSS,100.0
1,MODELO,100.0


## Cerrar conexión

In [14]:
conn.close()

print("Conexión cerrada")

Conexión cerrada
